# Triage Pilot — Phase 0 Checkpoint 4: Gemma 4 E4B Baseline

**Purpose:** Zero-shot Gemma 4 E4B across the frozen test set for all four SAM tasks. Produces the accuracy / F1 / latency / VRAM numbers that SAMs will be measured against.

**Runs on:** Vertex AI with A100 (preferred) or L4. E4B is too large for a T4 in FP16 without heavy quantization — use Vertex for this one.

**Output:** writes results into `baselines/baseline_results.json` under the key `gemma_e4b`.

## 1. Config

In [ ]:
import os

# Adjust if running on Vertex with a different mount
TRIAGE_ROOT = '/content/drive/MyDrive/setique/triage'
DATA_PROCESSED_DIR = f'{TRIAGE_ROOT}/data/processed'
EVAL_DIR = f'{TRIAGE_ROOT}/eval'
BASELINES_DIR = f'{TRIAGE_ROOT}/baselines'
PROMPTS_DIR = f'{BASELINES_DIR}/prompts'

TEST_PATH = f'{DATA_PROCESSED_DIR}/test.jsonl'
HASH_PATH = f'{EVAL_DIR}/test_set_hash.txt'
RESULTS_PATH = f'{BASELINES_DIR}/baseline_results.json'

MODEL_KEY = 'gemma_e4b'
# Update the HF model ID once the Gemma 4 E4B release name is confirmed on the hub
MODEL_ID = 'google/gemma-4-4b-it'  # ← placeholder; verify before running

# Set to a small int for a smoke test run; None for full test set
LIMIT = None

## 2. Mount + imports

In [ ]:
# Drive mount (comment out if running on Vertex with a different filesystem)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Not running in Colab — assuming TRIAGE_ROOT is already accessible.')

!pip install -q transformers accelerate bitsandbytes scikit-learn

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())  # assumes the notebook is launched from the repo root

from baselines.runner import (
    verify_test_hash, load_test_set, load_prompt,
    run_inference, merge_results,
    peak_vram_gb, reset_vram_tracking,
)

## 3. Verify frozen test set

In [ ]:
test_hash = verify_test_hash(TEST_PATH, HASH_PATH)
records = load_test_set(TEST_PATH)
print(f'Test hash verified: {test_hash}')
print(f'Loaded {len(records)} test records')

## 4. Load Gemma 4 E4B

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

reset_vram_tracking()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# Prefer bf16 on A100/L4; fall back to fp16 if the device doesn't support it
preferred_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=preferred_dtype,
    device_map='auto',
)
model.eval()
print(f'Loaded {MODEL_ID} in {preferred_dtype}. Peak VRAM after load: {peak_vram_gb():.2f} GB')

## 5. Build the generate function

Greedy decode with a tight max_new_tokens — all four SAMs produce short label outputs, so we don't need sampling.

In [ ]:
@torch.inference_mode()
def generate(prompt: str, max_new_tokens: int = 16) -> str:
    # Use chat template if the model supports one; otherwise raw prompt.
    messages = [{'role': 'user', 'content': prompt}]
    try:
        input_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors='pt'
        ).to(model.device)
    except Exception:
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(model.device)

    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Strip the prompt portion — only return the generated continuation
    new_tokens = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Sanity check
print('Smoke test output:', generate('Respond with the single word: ready'))

## 6. Run all four SAM tasks

Each task reuses the same frozen test records, scored against a different `target_field` and a different frozen prompt.

In [ ]:
# Label sets (mirror what's in the eval freeze notebook)
LANG_CODES = {'en','es','fr','de','pt','it','nl','ja','zh','ko',
              'ar','ru','pl','tr','sv','hi','id','vi','th','other'}
INTENT_LABELS = {
    'billing_question','billing_dispute','refund_request','cancel_subscription',
    'password_reset','account_access','update_profile','delete_account',
    'how_to','feature_request','bug_report','compatibility_question',
    'order_status','shipping_question','return_request','damaged_item',
    'technical_issue','escalation_request','complaint','compliment',
    'spam','unclear','other',
}
PRIORITY_LABELS = {'P1','P2','P3','P4'}
ROUTE_LABELS = {
    'billing_team','technical_support','account_security',
    'returns_and_shipping','product_feedback','legal_compliance',
    'human_escalation','auto_close',
}

TASKS = [
    ('sam_lang',     'lang',     LANG_CODES,     'other'),
    ('sam_intent',   'intent',   INTENT_LABELS,  'other'),
    ('sam_priority', 'priority', PRIORITY_LABELS,'P3'),
    ('sam_route',    'route',    ROUTE_LABELS,   'human_escalation'),
]

results_per_sam = {}
for sam_name, target_field, valid_labels, fallback in TASKS:
    print(f'\n=== {sam_name} ===')
    prompt_path = f'{PROMPTS_DIR}/{sam_name}.txt'
    prompt_template = load_prompt(prompt_path)
    reset_vram_tracking()
    result = run_inference(
        records=records,
        prompt_template=prompt_template,
        generate_fn=generate,
        valid_labels=valid_labels,
        target_field=target_field,
        fallback=fallback,
        limit=LIMIT,
        verbose=True,
    )
    result['peak_vram_gb'] = peak_vram_gb()
    results_per_sam[sam_name] = result
    print(f'  accuracy:   {result["accuracy"]:.4f}')
    print(f'  f1_macro:   {result["f1_macro"]:.4f}')
    print(f'  latency p50:{result["latency_ms_p50"]:.0f} ms')
    print(f'  latency p95:{result["latency_ms_p95"]:.0f} ms')
    print(f'  fallback %: {result["parse_fallback_rate"]*100:.2f}%')
    print(f'  peak VRAM:  {result["peak_vram_gb"]:.2f} GB')

## 7. Persist results

In [ ]:
from datetime import datetime

metadata = {
    'model_id': MODEL_ID,
    'dtype': str(preferred_dtype).split('.')[-1],
    'test_hash': test_hash,
    'n_test': len(records),
    'limit': LIMIT,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'decode': 'greedy, max_new_tokens=16',
}

merge_results(RESULTS_PATH, MODEL_KEY, results_per_sam, metadata)
print(f'Results merged into {RESULTS_PATH} under key "{MODEL_KEY}"')

## 8. Cost estimate

Rough per-1K-inference cost on whatever instance you're using. Update the hourly rate to match your Vertex instance.

In [ ]:
# Edit to match your actual Vertex instance hourly cost
INSTANCE_HOURLY_USD = 3.67  # placeholder for A100 40GB

for sam_name, res in results_per_sam.items():
    mean_sec = res['latency_ms_mean'] / 1000.0
    cost_per_1k = mean_sec * 1000 / 3600 * INSTANCE_HOURLY_USD
    print(f'{sam_name:14s}  ${cost_per_1k:.4f} per 1K inferences')

## 9. Checkpoint 4 verdict for E4B

- [ ] All four SAMs have recorded metrics
- [ ] `parse_fallback_rate` is reasonable (<10% ideally; >25% means the prompt is not working)
- [ ] `baseline_results.json` committed to the repo with the `gemma_e4b` key populated
- [ ] Run the E2B baseline notebook next to populate `gemma_e2b`